In [1]:
#EX1
from random import seed
from random import random

# Initialize a network
def initialize_network(n_inputs, n_hidden, n_outputs):
	network = list()
	hidden_layer = [{'weights':[round(random(),1) for i in range(n_inputs + 1)]} for i in range(n_hidden)]
	network.append(hidden_layer)
	output_layer = [{'weights':[round(random(),1) for i in range(n_hidden + 1)]} for i in range(n_outputs)]
	network.append(output_layer)
	return network 
  ######

def write(network):
    for i in range(len(network)):
        layer = network[i]
        print('layer :',i+1,'\n')
        for neuron in layer:
	        print(neuron,'\n')

In [ ]:
#Extention au multicouches 

def initialize_network(n_inputs, n_hidden, n_outputs):
    """
    n_inputs  : int
    n_hidden  : int  (mono-couche cachée, comportement d'origine)
               ou list/tuple d'int (multi-couches cachées)
    n_outputs : int
    """
    network = []

    # --- Cas 1 : ancien comportement (1 seule couche cachée) ---
    if isinstance(n_hidden, int):
        hidden_layer = [{'weights': [round(random(), 1) for _ in range(n_inputs + 1)]}
                        for _ in range(n_hidden)]
        network.append(hidden_layer)
        output_layer = [{'weights': [round(random(), 1) for _ in range(n_hidden + 1)]}
                        for _ in range(n_outputs)]
        network.append(output_layer)
        return network

    # --- Cas 2 : nouveau (multi-couches cachées) ---
    # n_hidden = [h1, h2, ..., hk]
    prev_size = n_inputs
    for h in n_hidden:
        hidden_layer = [{'weights': [round(random(), 1) for _ in range(prev_size + 1)]}
                        for _ in range(h)]
        network.append(hidden_layer)
        prev_size = h

    # Couche de sortie
    output_layer = [{'weights': [round(random(), 1) for _ in range(prev_size + 1)]}
                    for _ in range(n_outputs)]
    network.append(output_layer)
    return network

In [ ]:
def activate(weights, inputs):
	activation = weights[-1]
	for i in range(len(weights)-1):
		activation += weights[i] * inputs[i]
	return activation


In [3]:
from math import exp, tanh as _py_tanh, log

# --- Activations (scalaires) ---
def _act_linear(x):
    return x

def _act_sigmoid(x):
    return 1.0 / (1.0 + exp(-x))

def _act_tanh(x):
    return _py_tanh(x)

def _act_relu(x):
    return x if x > 0 else 0.0

def get_activation_fn(name):
    name = (name or 'sigmoid').lower()
    if name == 'linear':
        return _act_linear
    if name == 'relu':
        return _act_relu
    if name in ('sigmoid','logistic'):
        return _act_sigmoid
    if name in ('tanh','tangh','tanhg'):
        return _act_tanh
    # 'softmax' is handled at layer-level in forward_propagate
    return _act_sigmoid

# Compatibilité descendante : transfer = sigmoïde
def transfer(activation):
    return _act_sigmoid(activation)

  
  
  
# Forward propagate input to a network output
# 'activations' : liste de noms (relu, linear, sigmoid, tanh, softmax) une par couche
def forward_propagate(network, row, activations=None):
    inputs = row
    n_layers = len(network)
    if activations is None:
        activations = ['sigmoid'] * n_layers
    assert len(activations) == n_layers, "len(activations) must equal number of layers"
    for li, layer in enumerate(network):
        layer_act = (activations[li] or 'sigmoid').lower()
        # calc pre-activations pour toute la couche
        zs = []
        for neuron in layer:
            z = activate(neuron['weights'], inputs)
            zs.append(z)
        # appliquer activation couche
        if layer_act == 'softmax':
            # stable softmax
            m = max(zs)
            exps = [exp(z - m) for z in zs]
            s = sum(exps) if exps else 1.0
            outs = [e/s for e in exps]
        else:
            act_fn = get_activation_fn(layer_act)
            outs = [act_fn(z) for z in zs]
        # écrire outputs dans neurones et passer à la suite
        for neuron, y in zip(layer, outs):
            neuron['output'] = y
        inputs = outs
    return inputs